# Phase 2: Seed sweep (fixed schedule) — Colab

**目的**: ローカルのパイロット学習で、$V_{max}(t)=F(t)/k(t)$ の収束先が学習ごとに大きくばらついた(約30〜40)。エポック数などのスケジュールを揃えずにseedだけ変えていたため、「学習不足」と「本質的な識別不能性」のどちらが原因か切り分けられていなかった。このノートブックは**スケジュールを固定してseedだけを複数試し**、どちらが正しいかを確認する。

詳細: `documents/phase2_pilot_results.md`、実行するロジックは `scripts/phase2_seed_sweep.py`。

**判定基準**: seed間の $V_{max}$ プラトーの変動係数(CV) が10%未満なら「収束していた、スケジュールの問題だった」、それ以上なら「まだ不安定=識別性に課題が残る」。

**実行前に**: メニューの `ランタイム > ランタイムのタイプを変更` で **GPU** を選択してください。

## 1. Google Drive をマウント(結果の永続化用)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = '/content/drive/MyDrive/pi-fsm/phase2_seed_sweep'
import os
os.makedirs(OUT_DIR, exist_ok=True)
print('results will be saved to', OUT_DIR)

## 2. リポジトリを取得

既にcloneしてあれば `git pull` で最新化する。特定のコミット/タグに固定したい場合は `git checkout <tag>` を追加。

In [ ]:
%cd /content
if not os.path.exists('/content/pi-fsm'):
    !git clone https://github.com/ryu622/pi-fsm.git
%cd /content/pi-fsm
!git pull
!git log --oneline -5

## 3. 依存関係のインストール

Colabには torch/numpy/pandas/matplotlib/scipy は概ね入っているが、`kloppy`, `mplsoccer`, `pyarrow` 等は入っていないことが多い。`pip install -e .` で `pyproject.toml` の依存関係を一括インストールする(数分かかることがある)。

In [ ]:
!pip install -q -e .

import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected — go to ランタイム > ランタイムのタイプを変更 > GPU')

## 4. Seedスウィープ実行

1試合(J03WPY)、固定スケジュール、5 seed。GPUなら1seedあたり数分程度の見込み(ローカルCPU/MPSでは8〜30分かかっていた)。

In [ ]:
!python scripts/phase2_seed_sweep.py --match-id J03WPY --seeds 0,1,2,3,4 --out-dir "{OUT_DIR}"

## 5. 結果の確認

In [ ]:
import json
from IPython.display import Image, display

with open(f'{OUT_DIR}/summary.json') as f:
    summary = json.load(f)

print(f"Vmax plateau: mean={summary['vmax_plateau_mean']:.2f}, std={summary['vmax_plateau_std']:.2f}, cv={summary['vmax_plateau_cv']:.1%}")
print('Baseline1 (dt=1s) reference: Vmax=12.34')
verdict = '収束していた(スケジュールの問題だった)' if summary['vmax_plateau_cv'] < 0.10 else 'まだ不安定(識別性の課題が残る)'
print('判定:', verdict)

display(Image(f'{OUT_DIR}/seed_sweep_summary.png'))

## 次のステップ

- **CV<10%の場合**: スケジュール(エポック数・学習率)の問題だったと確定。あとはBaseline1との倍率のズレ(前回2.4倍)の原因を追う
- **CV>=10%の場合**: 識別性の課題が本質的に残っている。$v_0$ レンジを広げる(区間の開始条件を緩める)ことと短区間制限を**両方同時に**試す、あるいは選手embeddingなどの追加の条件付けを検討する
- 結果一式(`summary.json`, `seed_*.json`, 図)はDriveの `{OUT_DIR}` に永続化されているので、Colabセッションが切れても失われない

## 6. 追加実験: v0レンジを広げて同時に試す(CV>=10%だった場合)

結果: **CV=31.8%(mean=47.09, std=14.99)— 識別性の課題が残っていることを確認**。

以前 $v_0$ レンジを広げる実験(`v_low`=2.0→6.0)は単独では効果がなかったが、その時は区間を短く絞る前だった(まだ$\mathcal{L}_{data}$が収束していない設定と混ざっていた)。今度は**短区間制限(t≤2.5s、既に組み込み済み)と広い$v_0$レンジを同時に**試す。`--v-low` オプションで区間開始条件を緩める。

まず新しいセルをこのノートブックに追加して(下のコードをコピー)実行:

In [ ]:
%cd /content/pi-fsm
!git pull

OUT_DIR_WIDE = '/content/drive/MyDrive/pi-fsm/phase2_seed_sweep_wide_v0'
!python scripts/phase2_seed_sweep.py --match-id J03WPY --seeds 0,1,2,3,4 --v-low 6.0 --out-dir "{OUT_DIR_WIDE}"

In [ ]:
with open(f'{OUT_DIR_WIDE}/summary.json') as f:
    summary_wide = json.load(f)

print(f"[wide v0] Vmax plateau: mean={summary_wide['vmax_plateau_mean']:.2f}, std={summary_wide['vmax_plateau_std']:.2f}, cv={summary_wide['vmax_plateau_cv']:.1%}")
print(f"[narrow v0, previous] cv={summary['vmax_plateau_cv']:.1%}")
print('Baseline1 (dt=1s) reference: Vmax=12.34')
verdict_wide = '改善した(v0多様性が効いていた)' if summary_wide['vmax_plateau_cv'] < summary['vmax_plateau_cv'] * 0.7 else '改善しなかった(v0レンジは主因ではない)'
print('判定:', verdict_wide)

display(Image(f'{OUT_DIR_WIDE}/seed_sweep_summary.png'))